# Download Datasets

Download benchmark `.h5ad` files into the project-level `data/` directory. The default selection downloads the pancreas dataset used by the analysis notebooks.

In [ ]:
from pathlib import Path


# Resolve paths whether Jupyter starts in the project root or in notebooks/.
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
data_dir = project_root / "data"
data_dir.mkdir(exist_ok=True)

project_root, data_dir

In [ ]:
DATASETS = {
    # Main real-batch benchmark used by the SCALP-lite notebooks.
    "pancreas": {
        "kind": "url",
        "filename": "pancreas_normalized.h5ad",
        "url": "https://zenodo.org/records/3930949/files/pancreas_normalized.h5ad?download=1",
        "md5": "9b8bdaff49978c661f460f41afcfe0e0",
        "batch_key": "study",
        "label_key": "cell_type",
        "source": "https://zenodo.org/records/3930949",
    },
    # Small smoke-test dataset. It has no true batch, so downstream notebooks create artificial batches if needed.
    "pbmc3k": {
        "kind": "figshare",
        "filename": "scanpy-pbmc3k.h5ad",
        "article_id": 16447278,
        "preferred_file": "scanpy-pbmc3k.h5ad",
        "batch_key": "batch",
        "label_key": "leiden",
        "source": "https://figshare.com/articles/dataset/scanpy-pbmc3k_h5ad/16447278",
    },
    # Optional CellRank developmental example. Requires `pip install cellrank`.
    "zebrafish": {
        "kind": "cellrank",
        "filename": "cellrank-zebrafish.h5ad",
        "function": "zebrafish",
        "batch_key": "Stage",
        "label_key": "lineages",
        "source": "https://cellrank.readthedocs.io/en/latest/api/_autosummary/datasets/cellrank.datasets.zebrafish.html",
    },
    # Optional CellRank pancreas example. Requires `pip install cellrank`.
    "cellrank_pancreas": {
        "kind": "cellrank",
        "filename": "cellrank-pancreas.h5ad",
        "function": "pancreas",
        "batch_key": "clusters",
        "label_key": "clusters",
        "source": "https://cellrank.readthedocs.io/en/latest/api/_autosummary/datasets/cellrank.datasets.pancreas.html",
    },
}

# Edit this list to download more datasets.
selected_datasets = ["pancreas"]
# selected_datasets = ["pancreas", "pbmc3k"]
# selected_datasets = ["pancreas", "pbmc3k", "zebrafish", "cellrank_pancreas"]

[(name, DATASETS[name]["filename"]) for name in selected_datasets]

In [ ]:
import hashlib
import json
import shutil
import urllib.request


def md5sum(path, chunk_size=1024 * 1024):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_file(url, output_path, *, expected_md5=None, overwrite=False, chunk_size=1024 * 1024):
    output_path = Path(output_path)
    if output_path.exists() and not overwrite:
        if expected_md5 is None or md5sum(output_path) == expected_md5:
            print(f"exists: {output_path}")
            return output_path
        raise ValueError(f"Existing file has wrong md5: {output_path}")

    tmp_path = output_path.with_suffix(output_path.suffix + ".download")
    if tmp_path.exists():
        tmp_path.unlink()

    print(f"downloading: {url}")
    with urllib.request.urlopen(url) as response, open(tmp_path, "wb") as handle:
        shutil.copyfileobj(response, handle, length=chunk_size)

    if expected_md5 is not None:
        observed_md5 = md5sum(tmp_path)
        if observed_md5 != expected_md5:
            tmp_path.unlink(missing_ok=True)
            raise ValueError(f"md5 mismatch for {output_path.name}: {observed_md5} != {expected_md5}")

    tmp_path.replace(output_path)
    print(f"saved: {output_path}")
    return output_path


def figshare_download_url(article_id, *, preferred_file=None):
    api_url = f"https://api.figshare.com/v2/articles/{article_id}"
    with urllib.request.urlopen(api_url) as response:
        article = json.load(response)
    files = article.get("files", [])
    if preferred_file is not None:
        for file_info in files:
            if file_info.get("name") == preferred_file:
                return file_info["download_url"]
    if len(files) == 1:
        return files[0]["download_url"]
    names = [file_info.get("name") for file_info in files]
    raise ValueError(f"Could not choose a Figshare file from: {names}")


def download_cellrank_dataset(function_name, output_path, *, overwrite=False):
    output_path = Path(output_path)
    if output_path.exists() and not overwrite:
        print(f"exists: {output_path}")
        return output_path
    try:
        import cellrank as cr
    except ImportError as exc:
        raise ImportError("CellRank datasets require `pip install cellrank`.") from exc

    dataset_function = getattr(cr.datasets, function_name)
    dataset_function(path=str(output_path))
    print(f"saved: {output_path}")
    return output_path

In [ ]:
downloaded = {}
overwrite = False

for name in selected_datasets:
    dataset = DATASETS[name]
    output_path = data_dir / dataset["filename"]

    if dataset["kind"] == "url":
        path = download_file(
            dataset["url"],
            output_path,
            expected_md5=dataset.get("md5"),
            overwrite=overwrite,
        )
    elif dataset["kind"] == "figshare":
        url = figshare_download_url(dataset["article_id"], preferred_file=dataset.get("preferred_file"))
        path = download_file(url, output_path, overwrite=overwrite)
    elif dataset["kind"] == "cellrank":
        path = download_cellrank_dataset(dataset["function"], output_path, overwrite=overwrite)
    else:
        raise ValueError(f"Unknown dataset kind: {dataset['kind']}")

    downloaded[name] = path

downloaded

In [ ]:
import anndata as ad
import pandas as pd


rows = []
for name, path in downloaded.items():
    dataset = DATASETS[name]
    adata = ad.read_h5ad(path, backed="r")
    rows.append(
        {
            "dataset": name,
            "path": str(path.relative_to(project_root)),
            "cells": adata.n_obs,
            "genes": adata.n_vars,
            "batch_key": dataset["batch_key"],
            "has_batch_key": dataset["batch_key"] in adata.obs,
            "label_key": dataset["label_key"],
            "has_label_key": dataset["label_key"] in adata.obs,
            "source": dataset["source"],
        }
    )
    adata.file.close()

pd.DataFrame(rows)